# 结构化输出 structured_output
models 模型可以请求LLM提供格式化后的响应，通过给一个 schema。
这对于确保输出是被简单切分过得，可以直接被使用在后续的过程中。
LangChain提供了多个  schema 类型和方法来强制结构化的输出。

先创建一个Model

In [1]:
import os

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

## Pydantic
Pydantic Models 提供了最丰富的特性通过字段校验，描述和嵌套结构。

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")


In [ ]:
from langchain_core.messages import HumanMessage
model_with_structure = model.with_structured_output(Movie)

# ✅ 强制模型返回完整JSON（关键！） 为了适配千问，其中之一的解决办法：
# 提示词必须强制要求返回所有字段
# 必须包含 Return in valid JSON format.
prompt = """
Extract the information about the movie 'Inception'.
You MUST return a valid JSON object with ALL fields:
- title: string
- year: integer (only number)
- director: string
- rating: FLOAT (number between 0-10, like 8.2 or 9.0)
DO NOT return content rating like PG-13.
Return ONLY JSON.
"""

response = model_with_structure.invoke([
    HumanMessage(content=prompt)
])
print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8.8
